# 01 — IndicConformer 600M ASR — Clean Live Notebook

**Goal:** Hindi/Urdu speech → accurate transcript.

**Pipeline:** Microphone → 16 kHz normalization → RMS VAD → unlimited utterance accumulation → 5 seconds of continuous silence → IndicConformer CTC/RNNT → persistent WAV + JSONL/CSV audit.

There is **no maximum speech duration**. CTC and RNNT remain selectable for later comparison.

This is the cleaned notebook: one model load, one ASR function, one VAD+ASR callback, one Gradio app.

In [1]:
# 0. Verify Colab GPU
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime → Change runtime type → GPU"
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    f"VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [2]:
# 1. Install dependencies
!pip -q install -U transformers torchaudio soundfile gradio huggingface_hub onnxruntime

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 61.7 MB/s eta 0:00:00
✅ Dependencies installed


In [3]:
# 2. Hugging Face authentication + model loading
from huggingface_hub import login

login()

In [4]:
from transformers import AutoModel

MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
DEVICE = "cuda"

print("Loading model...")
t0 = time.perf_counter()

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
).to(DEVICE)

model.eval()

print(
    f"✅ Loaded in "
    f"{time.perf_counter() - t0:.2f}s"
)

Loading model...


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


✅ Loaded in 44.14s


In [5]:
# ============================================================
# LIVE TRANSCRIPTION — CLEAN WORKING EXPERIMENT
# IndicConformer 600M + Gradio Microphone Streaming
# ============================================================
#
# REQUIREMENTS:
#   - `model` must already be loaded
#   - `DEVICE` must already be defined
#
# IMPORTANT:
#   This is an isolated live-transcription implementation.
#   It does NOT modify the existing ASR pipeline.
#
# ARCHITECTURE:
#
#   Microphone
#       ↓
#   Gradio streaming chunks
#       ↓
#   Accumulated rolling audio
#       ↓
#   IndicConformer 600M RNNT
#       ↓
#   Live transcript
#
# The current IndicConformer ONNX encoder does not expose an
# encoder cache/state, so recent audio context is repeatedly
# encoded. This is intentionally kept simple and reliable.
# ============================================================


import gradio as gr
import numpy as np
import torch
import librosa


# ============================================================
# CONFIGURATION
# ============================================================

LIVE_SAMPLE_RATE = 16000

# How frequently Gradio sends audio to Python.
LIVE_STREAM_EVERY = 1.0

# Amount of recent audio supplied to IndicConformer.
#
# This is the rolling context, NOT the silence threshold.
LIVE_CONTEXT_SECONDS = 5.0

# Don't run ASR until enough audio has accumulated.
LIVE_MIN_AUDIO_SECONDS = 0.8



# ============================================================
# LIVE STATE
# ============================================================

def make_live_state():
    return {
        "audio": np.zeros(
            0,
            dtype=np.float32
        ),

        "transcript": "",

        "chunk_count": 0,
    }


# ============================================================
# AUDIO NORMALIZATION
# ============================================================

def normalize_live_audio(audio):

    if audio is None:
        return None

    # Gradio normally gives:
    # (sample_rate, numpy_array)
    if isinstance(audio, tuple):

        sample_rate, data = audio

    else:

        sample_rate = LIVE_SAMPLE_RATE
        data = audio

    if data is None:
        return None

    data = np.asarray(data)

    # --------------------------------------------------------
    # Stereo → mono
    # --------------------------------------------------------

    if data.ndim > 1:

        data = np.mean(
            data,
            axis=1
        )

    # --------------------------------------------------------
    # Convert PCM integer → float32
    # --------------------------------------------------------

    if np.issubdtype(
        data.dtype,
        np.integer
    ):

        info = np.iinfo(data.dtype)

        data = (
            data.astype(np.float32)
            / max(
                abs(info.min),
                info.max
            )
        )

    else:

        data = data.astype(
            np.float32
        )

    # --------------------------------------------------------
    # Resample → 16 kHz
    # --------------------------------------------------------

    if sample_rate != LIVE_SAMPLE_RATE:

        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=LIVE_SAMPLE_RATE
        )

        data = data.astype(
            np.float32
        )

    return data


# ============================================================
# LIVE ASR
# ============================================================

def run_live_asr(audio):

    if audio is None:
        return ""

    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS *
        LIVE_SAMPLE_RATE
    )

    if len(audio) < minimum_samples:
        return ""

    # --------------------------------------------------------
    # NumPy → Torch
    # --------------------------------------------------------

    wav = torch.from_numpy(
        audio
    ).float().unsqueeze(0).to(DEVICE)

    # --------------------------------------------------------
    # IndicConformer RNNT
    # --------------------------------------------------------

    with torch.inference_mode():

        text = model(
            wav,
            "hi",
            "rnnt"
        )

    if text is None:
        return ""

    return str(text).strip()


# ============================================================
# STREAMING CALLBACK
# ============================================================

def live_transcription(
    audio,
    state
):

    # --------------------------------------------------------
    # Initialize state
    # --------------------------------------------------------

    if state is None:

        state = make_live_state()


    # --------------------------------------------------------
    # No audio
    # --------------------------------------------------------

    if audio is None:

        return (
            state["transcript"],
            state
        )


    # --------------------------------------------------------
    # Normalize incoming chunk
    # --------------------------------------------------------

    chunk = normalize_live_audio(
        audio
    )

    if chunk is None or len(chunk) == 0:

        return (
            state["transcript"],
            state
        )


    # --------------------------------------------------------
    # Add chunk to rolling audio buffer
    # --------------------------------------------------------

    state["audio"] = np.concatenate(
        [
            state["audio"],
            chunk
        ]
    )

    state["chunk_count"] += 1


    # --------------------------------------------------------
    # Keep only recent context
    # --------------------------------------------------------

    max_samples = int(
        LIVE_CONTEXT_SECONDS *
        LIVE_SAMPLE_RATE
    )

    if len(state["audio"]) > max_samples:

        state["audio"] = state["audio"][
            -max_samples:
        ]


    # --------------------------------------------------------
    # Run ASR
    # --------------------------------------------------------

    try:

        text = run_live_asr(
            state["audio"]
        )

        if text:

            state["transcript"] = text

    except Exception as e:

        state["transcript"] = (
            f"ASR ERROR: "
            f"{type(e).__name__}: {e}"
        )


    # --------------------------------------------------------
    # Return live transcript
    # --------------------------------------------------------

    return (
        state["transcript"],
        state
    )


# ============================================================
# GRADIO LIVE INTERFACE
# ============================================================

# with gr.Blocks(
#     title="Live IndicConformer Transcription"
# ) as live_demo:

#     gr.Markdown(
#         """
#         # 🎙️ Live Transcription

#         Speak continuously and the transcript will update
#         automatically.

#         **IndicConformer 600M · RNNT**
#         """
#     )

#     microphone = gr.Audio(
#         sources=["microphone"],
#         type="numpy",
#         streaming=True,
#         label="🎤 Microphone"
#     )

#     transcript = gr.Textbox(
#         label="Live Transcript",
#         lines=12,
#         interactive=False
#     )

#     live_state = gr.State(
#         make_live_state()
#     )

#     microphone.stream(
#         fn=live_transcription,
#         inputs=[
#             microphone,
#             live_state
#         ],
#         outputs=[
#             transcript,
#             live_state
#         ],
#         stream_every=LIVE_STREAM_EVERY,
#     )


# # ============================================================
# # LAUNCH
# # ============================================================

# live_demo.launch(
#     share=True,
#     debug=True
# )

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3541ca092f8fa6bb78.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3541ca092f8fa6bb78.gradio.live


In [6]:
# ============================================================
# LIVE TRANSCRIPTION — STEP 2
# COMMITTED + PARTIAL TEXT
# ============================================================
#
# This builds directly on the successful live ASR experiment.
#
# Nothing above is modified.
#
# Goal:
#
#     COMMITTED TEXT
#     +
#     CURRENT PARTIAL TEXT
#
# Example:
#
#     Committed:
#     "आज मौसम बहुत अच्छा है"
#
#     Partial:
#     "और मैं बाहर"
#
#     Display:
#     "आज मौसम बहुत अच्छा है और मैं बाहर"
#
# ============================================================


import gradio as gr
import numpy as np
import torch
import librosa


# ============================================================
# CONFIGURATION
# ============================================================

LIVE_SAMPLE_RATE = 16000

# Gradio sends approximately one chunk this often.
LIVE_STREAM_EVERY = 1.0

# Rolling context supplied to IndicConformer.
LIVE_CONTEXT_SECONDS = 5.0

# Minimum audio before ASR.
LIVE_MIN_AUDIO_SECONDS = 0.8

# ------------------------------------------------------------
# IMPORTANT
#
# We deliberately use a SHORT silence threshold here.
#
# This is NOT required for transcription.
# It only decides when the current partial text should become
# committed text.
#
# We will tune this later.
# ------------------------------------------------------------

PARTIAL_COMMIT_SILENCE = 1.5


# ============================================================
# STATE
# ============================================================

def make_live_state_v2():

    return {

        # Recent audio supplied to ASR
        "audio": np.zeros(
            0,
            dtype=np.float32
        ),

        # Text that has been committed
        "committed_transcript": "",

        # Current rolling ASR result
        "partial_transcript": "",

        # Silence tracking
        "silence_duration": 0.0,

        # Debug / tracking
        "chunk_count": 0,
    }


# ============================================================
# AUDIO NORMALIZATION
# ============================================================

def normalize_live_audio_v2(audio):

    if audio is None:
        return None

    # Gradio streaming format
    if isinstance(audio, tuple):

        sample_rate, data = audio

    else:

        sample_rate = LIVE_SAMPLE_RATE
        data = audio


    if data is None:
        return None


    data = np.asarray(data)


    # --------------------------------------------------------
    # Stereo → mono
    # --------------------------------------------------------

    if data.ndim > 1:

        data = np.mean(
            data,
            axis=1
        )


    # --------------------------------------------------------
    # PCM → float32
    # --------------------------------------------------------

    if np.issubdtype(
        data.dtype,
        np.integer
    ):

        info = np.iinfo(data.dtype)

        data = (
            data.astype(np.float32)
            / max(
                abs(info.min),
                info.max
            )
        )

    else:

        data = data.astype(
            np.float32
        )


    # --------------------------------------------------------
    # Resample
    # --------------------------------------------------------

    if sample_rate != LIVE_SAMPLE_RATE:

        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=LIVE_SAMPLE_RATE
        )

        data = data.astype(
            np.float32
        )


    return data


# ============================================================
# SIMPLE ENERGY DETECTOR
# ============================================================

def calculate_rms(audio):

    if audio is None or len(audio) == 0:
        return 0.0

    return float(
        np.sqrt(
            np.mean(
                np.square(
                    audio
                )
            )
        )
    )


def is_speech(audio):

    if audio is None or len(audio) == 0:
        return False

    # Same conservative threshold we already tested.
    return calculate_rms(audio) > 0.008


# ============================================================
# RUN INDICCONFORMER
# ============================================================

def run_live_asr_v2(audio):

    if audio is None:
        return ""

    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS *
        LIVE_SAMPLE_RATE
    )

    if len(audio) < minimum_samples:
        return ""


    wav = torch.from_numpy(
        audio
    ).float().unsqueeze(0).to(DEVICE)


    with torch.inference_mode():

        text = model(
            wav,
            "hi",
            "rnnt"
        )


    if text is None:
        return ""

    return str(text).strip()


# ============================================================
# BUILD DISPLAY TEXT
# ============================================================

def build_display_text(state):

    committed = (
        state["committed_transcript"]
        .strip()
    )

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if committed and partial:

        return (
            committed
            + " "
            + partial
        )

    if committed:

        return committed

    return partial


# ============================================================
# LIVE CALLBACK
# ============================================================

def live_transcription_v2(
    audio,
    state
):

    # --------------------------------------------------------
    # Initialize state
    # --------------------------------------------------------

    if state is None:

        state = make_live_state_v2()


    # --------------------------------------------------------
    # No audio
    # --------------------------------------------------------

    if audio is None:

        return (
            build_display_text(state),
            state
        )


    # --------------------------------------------------------
    # Normalize chunk
    # --------------------------------------------------------

    chunk = normalize_live_audio_v2(
        audio
    )

    if chunk is None or len(chunk) == 0:

        return (
            build_display_text(state),
            state
        )


    # --------------------------------------------------------
    # Track chunk
    # --------------------------------------------------------

    state["chunk_count"] += 1


    chunk_duration = (
        len(chunk)
        / LIVE_SAMPLE_RATE
    )


    # --------------------------------------------------------
    # Speech / silence
    # --------------------------------------------------------

    speech = is_speech(chunk)


    if speech:

        state["silence_duration"] = 0.0

    else:

        state["silence_duration"] += (
            chunk_duration
        )


    # --------------------------------------------------------
    # Add audio to rolling buffer
    # --------------------------------------------------------

    state["audio"] = np.concatenate(
        [
            state["audio"],
            chunk
        ]
    )


    # --------------------------------------------------------
    # Keep only recent context
    # --------------------------------------------------------

    max_samples = int(
        LIVE_CONTEXT_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(state["audio"]) > max_samples:

        state["audio"] = state["audio"][
            -max_samples:
        ]


    # --------------------------------------------------------
    # RUN ASR
    # --------------------------------------------------------

    try:

        current_text = run_live_asr_v2(
            state["audio"]
        )

        if current_text:

            state["partial_transcript"] = (
                current_text
            )


    except Exception as e:

        return (
            f"ASR ERROR: "
            f"{type(e).__name__}: {e}",
            state
        )


    # --------------------------------------------------------
    # COMMIT AFTER SILENCE
    # --------------------------------------------------------

    if (
        state["silence_duration"]
        >= PARTIAL_COMMIT_SILENCE
    ):

        if state["partial_transcript"]:

            if state["committed_transcript"]:

                state["committed_transcript"] += (
                    " "
                    + state["partial_transcript"]
                )

            else:

                state["committed_transcript"] = (
                    state["partial_transcript"]
                )


        # Reset current utterance
        state["partial_transcript"] = ""

        state["audio"] = np.zeros(
            0,
            dtype=np.float32
        )

        state["silence_duration"] = 0.0


    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    return (
        build_display_text(state),
        state
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

# with gr.Blocks(
#     title="IndicConformer Live Transcription — Step 2"
# ) as live_demo_v2:

#     gr.Markdown(
#         """
#         # 🎙️ Live IndicConformer Transcription

#         **Committed + Partial transcription**

#         Speak continuously. The current speech appears immediately,
#         while previous speech is preserved.
#         """
#     )


#     microphone = gr.Audio(
#         sources=["microphone"],
#         type="numpy",
#         streaming=True,
#         label="🎤 Microphone"
#     )


#     transcript = gr.Textbox(
#         label="Live Transcript",
#         lines=12,
#         interactive=False
#     )


#     live_state = gr.State(
#         make_live_state_v2()
#     )


#     microphone.stream(
#         fn=live_transcription_v2,
#         inputs=[
#             microphone,
#             live_state
#         ],
#         outputs=[
#             transcript,
#             live_state
#         ],
#         stream_every=LIVE_STREAM_EVERY,
#     )


# # ============================================================
# # LAUNCH
# # ============================================================

# live_demo_v2.launch(
#     share=True,
#     debug=True
# )

In [7]:
# ============================================================
# LIVE TRANSCRIPTION — STEP 3
# REDUCED REDUNDANT ASR COMPUTATION
# ============================================================
#
# Builds on the successful Step 2 implementation.
#
# Improvements:
#
#   1. Silence chunks do NOT trigger ASR.
#   2. ASR is run only after enough NEW audio arrives.
#   3. The rolling context is still retained.
#   4. Committed + partial transcription is preserved.
#
# IMPORTANT:
#   This is still rolling-window inference.
#   We are NOT pretending the ONNX encoder is stateful.
#
# ============================================================


import gradio as gr
import numpy as np
import torch
import librosa


# ============================================================
# CONFIGURATION
# ============================================================

LIVE_SAMPLE_RATE = 16000

# Gradio callback frequency
LIVE_STREAM_EVERY = 1.0

# Rolling context given to IndicConformer
LIVE_CONTEXT_SECONDS = 5.0

# Minimum total audio before first ASR
LIVE_MIN_AUDIO_SECONDS = 0.8

# ------------------------------------------------------------
# VAD
# ------------------------------------------------------------

LIVE_RMS_THRESHOLD = 0.008

# ------------------------------------------------------------
# ASR throttling
# ------------------------------------------------------------
#
# We do not need to run the 600M model for every tiny change.
#
# Example:
#
#   incoming audio
#        ↓
#   accumulate
#        ↓
#   0.5 sec new speech
#        ↓
#   ASR
#
# ------------------------------------------------------------

LIVE_NEW_AUDIO_FOR_ASR = 0.5

# ------------------------------------------------------------
# Silence used to commit the current partial result.
# ------------------------------------------------------------

PARTIAL_COMMIT_SILENCE = 1.5


# ============================================================
# STATE
# ============================================================

def make_live_state_v3():

    return {

        # ----------------------------------------------------
        # Rolling audio
        # ----------------------------------------------------

        "audio": np.zeros(
            0,
            dtype=np.float32
        ),

        # ----------------------------------------------------
        # Transcript
        # ----------------------------------------------------

        "committed_transcript": "",

        "partial_transcript": "",

        # ----------------------------------------------------
        # Timing
        # ----------------------------------------------------

        "silence_duration": 0.0,

        "new_audio_since_asr": 0.0,

        # ----------------------------------------------------
        # Debugging / statistics
        # ----------------------------------------------------

        "chunk_count": 0,

        "asr_count": 0,
    }


# ============================================================
# AUDIO NORMALIZATION
# ============================================================

def normalize_live_audio_v3(audio):

    if audio is None:
        return None


    # Gradio format:
    # (sample_rate, numpy_array)

    if isinstance(audio, tuple):

        sample_rate, data = audio

    else:

        sample_rate = LIVE_SAMPLE_RATE
        data = audio


    if data is None:
        return None


    data = np.asarray(data)


    # --------------------------------------------------------
    # Stereo → mono
    # --------------------------------------------------------

    if data.ndim > 1:

        data = np.mean(
            data,
            axis=1
        )


    # --------------------------------------------------------
    # PCM → float32
    # --------------------------------------------------------

    if np.issubdtype(
        data.dtype,
        np.integer
    ):

        info = np.iinfo(data.dtype)

        data = (
            data.astype(np.float32)
            / max(
                abs(info.min),
                info.max
            )
        )

    else:

        data = data.astype(
            np.float32
        )


    # --------------------------------------------------------
    # Resample
    # --------------------------------------------------------

    if sample_rate != LIVE_SAMPLE_RATE:

        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=LIVE_SAMPLE_RATE
        )

        data = data.astype(
            np.float32
        )


    return data


# ============================================================
# RMS VAD
# ============================================================

def calculate_rms_v3(audio):

    if audio is None or len(audio) == 0:

        return 0.0


    return float(
        np.sqrt(
            np.mean(
                np.square(
                    audio
                )
            )
        )
    )


def is_speech_v3(audio):

    if audio is None or len(audio) == 0:

        return False


    rms = calculate_rms_v3(
        audio
    )


    return rms > LIVE_RMS_THRESHOLD


# ============================================================
# INDICCONFORMER ASR
# ============================================================

def run_live_asr_v3(audio):

    if audio is None:
        return ""


    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(audio) < minimum_samples:

        return ""


    # --------------------------------------------------------
    # NumPy → Torch
    # --------------------------------------------------------

    wav = torch.from_numpy(
        audio
    ).float().unsqueeze(0).to(DEVICE)


    # --------------------------------------------------------
    # RNNT inference
    # --------------------------------------------------------

    with torch.inference_mode():

        text = model(
            wav,
            "hi",
            "rnnt"
        )


    if text is None:

        return ""


    return str(
        text
    ).strip()


# ============================================================
# DISPLAY
# ============================================================

def build_display_v3(state):

    committed = (
        state["committed_transcript"]
        .strip()
    )

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if committed and partial:

        return (
            committed
            + " "
            + partial
        )


    if committed:

        return committed


    return partial


# ============================================================
# COMMIT PARTIAL TRANSCRIPT
# ============================================================

def commit_partial_v3(state):

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if partial:

        if state["committed_transcript"]:

            state["committed_transcript"] += (
                " "
                + partial
            )

        else:

            state["committed_transcript"] = (
                partial
            )


    state["partial_transcript"] = ""


# ============================================================
# LIVE CALLBACK
# ============================================================

def live_transcription_v3(
    audio,
    state
):

    # --------------------------------------------------------
    # Initialize
    # --------------------------------------------------------

    if state is None:

        state = make_live_state_v3()


    # --------------------------------------------------------
    # No audio
    # --------------------------------------------------------

    if audio is None:

        return (
            build_display_v3(state),
            state
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    chunk = normalize_live_audio_v3(
        audio
    )


    if chunk is None or len(chunk) == 0:

        return (
            build_display_v3(state),
            state
        )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    state["chunk_count"] += 1


    chunk_duration = (
        len(chunk)
        / LIVE_SAMPLE_RATE
    )


    # --------------------------------------------------------
    # VAD
    # --------------------------------------------------------

    speech = is_speech_v3(
        chunk
    )


    # ========================================================
    # SPEECH
    # ========================================================

    if speech:

        # Speech means silence timer resets.

        state["silence_duration"] = 0.0


        # Add audio.

        state["audio"] = np.concatenate(
            [
                state["audio"],
                chunk
            ]
        )


        # Track NEW audio specifically.

        state["new_audio_since_asr"] += (
            chunk_duration
        )


    # ========================================================
    # SILENCE
    # ========================================================

    else:

        # ----------------------------------------------------
        # If we already have an active utterance, track silence.
        # ----------------------------------------------------

        if len(state["audio"]) > 0:

            state["silence_duration"] += (
                chunk_duration
            )


    # ========================================================
    # ROLLING CONTEXT
    # ========================================================

    max_samples = int(
        LIVE_CONTEXT_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(state["audio"]) > max_samples:

        state["audio"] = state["audio"][
            -max_samples:
        ]


    # ========================================================
    # ASR DECISION
    # ========================================================
    #
    # ASR runs when:
    #
    #   - enough NEW speech has arrived
    #
    # AND
    #
    #   - enough total audio exists
    #
    # Silence alone does not trigger ASR.
    #
    # ========================================================

    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    enough_audio = (
        len(state["audio"])
        >= minimum_samples
    )


    enough_new_audio = (
        state["new_audio_since_asr"]
        >= LIVE_NEW_AUDIO_FOR_ASR
    )


    should_run_asr = (
        enough_audio
        and
        enough_new_audio
        and
        speech
    )


    if should_run_asr:

        try:

            current_text = run_live_asr_v3(
                state["audio"]
            )


            if current_text:

                state["partial_transcript"] = (
                    current_text
                )


            state["asr_count"] += 1

            state["new_audio_since_asr"] = 0.0


        except Exception as e:

            return (
                f"ASR ERROR: "
                f"{type(e).__name__}: {e}",
                state
            )


    # ========================================================
    # COMMIT AFTER SILENCE
    # ========================================================

    if (
        state["silence_duration"]
        >= PARTIAL_COMMIT_SILENCE
    ):

        commit_partial_v3(
            state
        )


        # Clear rolling audio.

        state["audio"] = np.zeros(
            0,
            dtype=np.float32
        )


        # Reset counters.

        state["silence_duration"] = 0.0

        state["new_audio_since_asr"] = 0.0


    # ========================================================
    # RETURN
    # ========================================================

    return (
        build_display_v3(state),
        state
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

# with gr.Blocks(
#     title="IndicConformer Live Transcription — Step 3"
# ) as live_demo_v3:

#     gr.Markdown(
#         """
#         # 🎙️ Live IndicConformer Transcription

#         ### Step 3 — Efficient Rolling Context

#         Speak continuously and the transcript will update
#         while avoiding unnecessary ASR calls during silence.
#         """
#     )


#     microphone = gr.Audio(
#         sources=["microphone"],
#         type="numpy",
#         streaming=True,
#         label="🎤 Microphone"
#     )


#     transcript = gr.Textbox(
#         label="Live Transcript",
#         lines=12,
#         interactive=False
#     )


#     live_state = gr.State(
#         make_live_state_v3()
#     )


#     microphone.stream(
#         fn=live_transcription_v3,
#         inputs=[
#             microphone,
#             live_state
#         ],
#         outputs=[
#             transcript,
#             live_state
#         ],
#         stream_every=LIVE_STREAM_EVERY,
#     )


# # ============================================================
# # LAUNCH
# # ============================================================

# live_demo_v3.launch(
#     share=True,
#     debug=True
# )

In [11]:
# ============================================================
# LIVE TRANSCRIPTION — STEP 4
# ACTIVE UTTERANCE WINDOW
# ============================================================
#
# Builds on Step 3.
#
# IMPORTANT:
#   This does NOT assume that IndicConformer has a hidden
#   encoder cache. The exported ONNX interface does not expose
#   one.
#
# Instead:
#
#   COMMITTED TEXT
#          +
#   CURRENT UTTERANCE AUDIO
#          ↓
#   IndicConformer RNNT
#
# Once silence commits the utterance, the audio buffer is
# cleared. Therefore old conversation audio is never repeatedly
# re-encoded.
#
# ============================================================


import gradio as gr
import numpy as np
import torch
import librosa


# ============================================================
# CONFIGURATION
# ============================================================

LIVE_SAMPLE_RATE = 16000

# Gradio callback interval
LIVE_STREAM_EVERY = 1.0

# Maximum audio belonging to the currently active utterance.
#
# This prevents an extremely long speech segment from growing
# indefinitely.
LIVE_MAX_UTTERANCE_SECONDS = 8.0

# Minimum audio before ASR
LIVE_MIN_AUDIO_SECONDS = 0.8

# RMS threshold
LIVE_RMS_THRESHOLD = 0.008

# Run ASR after this much NEW speech.
LIVE_NEW_AUDIO_FOR_ASR = 0.5

# Silence required before committing the utterance.
LIVE_COMMIT_SILENCE = 1.5


# ============================================================
# STATE
# ============================================================

def make_live_state_v4():

    return {

        # ----------------------------------------------------
        # Current utterance audio ONLY
        # ----------------------------------------------------

        "audio": np.zeros(
            0,
            dtype=np.float32
        ),

        # ----------------------------------------------------
        # Transcript
        # ----------------------------------------------------

        "committed_transcript": "",

        "partial_transcript": "",

        # ----------------------------------------------------
        # Timing
        # ----------------------------------------------------

        "silence_duration": 0.0,

        "new_audio_since_asr": 0.0,

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        "chunk_count": 0,

        "asr_count": 0,

        "utterance_count": 0,
    }


# ============================================================
# AUDIO NORMALIZATION
# ============================================================

def normalize_audio_v4(audio):

    if audio is None:
        return None


    if isinstance(audio, tuple):

        sample_rate, data = audio

    else:

        sample_rate = LIVE_SAMPLE_RATE
        data = audio


    if data is None:
        return None


    data = np.asarray(data)


    # --------------------------------------------------------
    # Stereo → mono
    # --------------------------------------------------------

    if data.ndim > 1:

        data = np.mean(
            data,
            axis=1
        )


    # --------------------------------------------------------
    # Integer PCM → float32
    # --------------------------------------------------------

    if np.issubdtype(
        data.dtype,
        np.integer
    ):

        info = np.iinfo(data.dtype)

        data = (
            data.astype(np.float32)
            / max(
                abs(info.min),
                info.max
            )
        )

    else:

        data = data.astype(
            np.float32
        )


    # --------------------------------------------------------
    # Resample
    # --------------------------------------------------------

    if sample_rate != LIVE_SAMPLE_RATE:

        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=LIVE_SAMPLE_RATE
        )

        data = data.astype(
            np.float32
        )


    return data


# ============================================================
# VAD
# ============================================================

def rms_v4(audio):

    if audio is None or len(audio) == 0:

        return 0.0


    return float(
        np.sqrt(
            np.mean(
                np.square(
                    audio
                )
            )
        )
    )


def is_speech_v4(audio):

    if audio is None or len(audio) == 0:

        return False


    return (
        rms_v4(audio)
        > LIVE_RMS_THRESHOLD
    )


# ============================================================
# INDICCONFORMER
# ============================================================

def run_asr_v4(audio):

    if audio is None:

        return ""


    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(audio) < minimum_samples:

        return ""


    wav = torch.from_numpy(
        audio
    ).float().unsqueeze(0).to(DEVICE)


    with torch.inference_mode():

        text = model(
            wav,
            "hi",
            "rnnt"
        )


    if text is None:

        return ""


    return str(
        text
    ).strip()


# ============================================================
# DISPLAY
# ============================================================

def display_v4(state):

    committed = (
        state["committed_transcript"]
        .strip()
    )

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if committed and partial:

        return (
            committed
            + " "
            + partial
        )


    if committed:

        return committed


    return partial


# ============================================================
# COMMIT
# ============================================================

def commit_current_utterance(state):

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if partial:

        if state["committed_transcript"]:

            state["committed_transcript"] += (
                " "
                + partial
            )

        else:

            state["committed_transcript"] = (
                partial
            )


    state["partial_transcript"] = ""


    state["audio"] = np.zeros(
        0,
        dtype=np.float32
    )


    state["silence_duration"] = 0.0

    state["new_audio_since_asr"] = 0.0

    state["utterance_count"] += 1


# ============================================================
# LIVE CALLBACK
# ============================================================

def live_transcription_v4(
    audio,
    state
):

    if state is None:

        state = make_live_state_v4()


    # --------------------------------------------------------
    # No audio
    # --------------------------------------------------------

    if audio is None:

        return (
            display_v4(state),
            state
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    chunk = normalize_audio_v4(
        audio
    )


    if chunk is None or len(chunk) == 0:

        return (
            display_v4(state),
            state
        )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    state["chunk_count"] += 1


    chunk_duration = (
        len(chunk)
        / LIVE_SAMPLE_RATE
    )


    # --------------------------------------------------------
    # VAD
    # --------------------------------------------------------

    speech = is_speech_v4(
        chunk
    )


    # ========================================================
    # SPEECH
    # ========================================================

    if speech:

        # Reset silence.

        state["silence_duration"] = 0.0


        # Add to CURRENT utterance only.

        state["audio"] = np.concatenate(
            [
                state["audio"],
                chunk
            ]
        )


        # Track new speech.

        state["new_audio_since_asr"] += (
            chunk_duration
        )


    # ========================================================
    # SILENCE
    # ========================================================

    else:

        if len(state["audio"]) > 0:

            state["silence_duration"] += (
                chunk_duration
            )


    # ========================================================
    # PROTECT AGAINST VERY LONG UTTERANCES
    # ========================================================

    max_samples = int(
        LIVE_MAX_UTTERANCE_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(state["audio"]) > max_samples:

        state["audio"] = state["audio"][
            -max_samples:
        ]


    # ========================================================
    # ASR
    # ========================================================

    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    enough_audio = (
        len(state["audio"])
        >= minimum_samples
    )


    enough_new_audio = (
        state["new_audio_since_asr"]
        >= LIVE_NEW_AUDIO_FOR_ASR
    )


    should_run_asr = (
        speech
        and
        enough_audio
        and
        enough_new_audio
    )


    if should_run_asr:

        try:

            current_text = run_asr_v4(
                state["audio"]
            )


            if current_text:

                state["partial_transcript"] = (
                    current_text
                )


            state["asr_count"] += 1

            state["new_audio_since_asr"] = 0.0


        except Exception as e:

            return (
                f"ASR ERROR: "
                f"{type(e).__name__}: {e}",
                state
            )


    # ========================================================
    # COMMIT
    # ========================================================

    if (
        state["silence_duration"]
        >= LIVE_COMMIT_SILENCE
    ):

        commit_current_utterance(
            state
        )


    # ========================================================
    # RETURN
    # ========================================================

    return (
        display_v4(state),
        state
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="IndicConformer Live Transcription — Step 4"
) as live_demo_v4:

    gr.Markdown(
        """
        # 🎙️ Live IndicConformer Transcription

        ### Step 4 — Active Utterance Window

        Speak continuously.

        Previous utterances are committed and removed from
        the ASR audio buffer, while the current utterance
        remains available for incremental recognition.
        """
    )


    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="🎤 Microphone"
    )


    transcript = gr.Textbox(
        label="Live Transcript",
        lines=12,
        interactive=False
    )


    live_state = gr.State(
        make_live_state_v4()
    )


    microphone.stream(
        fn=live_transcription_v4,
        inputs=[
            microphone,
            live_state
        ],
        outputs=[
            transcript,
            live_state
        ],
        stream_every=LIVE_STREAM_EVERY,
    )


# ============================================================
# LAUNCH
# ============================================================

live_demo_v4.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://055b156a3836c828bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://055b156a3836c828bf.gradio.live


In [12]:
# ============================================================
# LIVE TRANSCRIPTION — STEP 5
# OVERLAPPING WINDOWS + TRANSCRIPT MERGING
# ============================================================
#
# Builds on the successful Step 4 experiment.
#
# NEW:
#
#   Long speech
#       ↓
#   overlapping audio windows
#       ↓
#   IndicConformer RNNT
#       ↓
#   overlap-aware text merge
#
# This allows a long continuous utterance to exceed the
# previous 8-second active-window limitation.
#
# IMPORTANT:
#   We still do NOT assume native encoder streaming.
#   Each active window is independently encoded.
#
# ============================================================


import gradio as gr
import numpy as np
import torch
import librosa


# ============================================================
# CONFIGURATION
# ============================================================

LIVE_SAMPLE_RATE = 16000

# Gradio callback interval
LIVE_STREAM_EVERY = 1.0

# Maximum audio passed to the model in one inference
WINDOW_SECONDS = 6.0

# Audio overlap between consecutive inference windows
OVERLAP_SECONDS = 2.0

# Therefore a new window normally advances by:
#
#   6 - 2 = 4 seconds
#
WINDOW_STEP_SECONDS = (
    WINDOW_SECONDS
    - OVERLAP_SECONDS
)

# Minimum audio before ASR
LIVE_MIN_AUDIO_SECONDS = 0.8

# VAD
LIVE_RMS_THRESHOLD = 0.008

# Minimum new speech before another ASR
LIVE_NEW_AUDIO_FOR_ASR = 0.5

# Silence that finalizes the utterance
LIVE_COMMIT_SILENCE = 1.5


# ============================================================
# STATE
# ============================================================

def make_live_state_v5():

    return {

        # ----------------------------------------------------
        # Entire CURRENT utterance audio.
        #
        # We retain enough history to create overlapping
        # windows.
        # ----------------------------------------------------

        "audio": np.zeros(
            0,
            dtype=np.float32
        ),

        # ----------------------------------------------------
        # Text already committed to the final transcript.
        # ----------------------------------------------------

        "committed_transcript": "",

        # ----------------------------------------------------
        # Most recent model output.
        # ----------------------------------------------------

        "partial_transcript": "",

        # ----------------------------------------------------
        # Last merged model output.
        #
        # This lets us compare consecutive overlapping
        # windows.
        # ----------------------------------------------------

        "last_window_transcript": "",

        # ----------------------------------------------------
        # Timing
        # ----------------------------------------------------

        "silence_duration": 0.0,

        "new_audio_since_asr": 0.0,

        # ----------------------------------------------------
        # Window bookkeeping
        # ----------------------------------------------------

        "window_count": 0,

        "last_window_start": 0.0,

        # ----------------------------------------------------
        # Debug statistics
        # ----------------------------------------------------

        "chunk_count": 0,

        "asr_count": 0,

        "utterance_count": 0,
    }


# ============================================================
# AUDIO NORMALIZATION
# ============================================================

def normalize_audio_v5(audio):

    if audio is None:
        return None


    if isinstance(audio, tuple):

        sample_rate, data = audio

    else:

        sample_rate = LIVE_SAMPLE_RATE
        data = audio


    if data is None:
        return None


    data = np.asarray(data)


    # --------------------------------------------------------
    # Stereo → mono
    # --------------------------------------------------------

    if data.ndim > 1:

        data = np.mean(
            data,
            axis=1
        )


    # --------------------------------------------------------
    # PCM → float32
    # --------------------------------------------------------

    if np.issubdtype(
        data.dtype,
        np.integer
    ):

        info = np.iinfo(data.dtype)

        data = (
            data.astype(np.float32)
            / max(
                abs(info.min),
                info.max
            )
        )

    else:

        data = data.astype(
            np.float32
        )


    # --------------------------------------------------------
    # Resample → 16 kHz
    # --------------------------------------------------------

    if sample_rate != LIVE_SAMPLE_RATE:

        data = librosa.resample(
            data,
            orig_sr=sample_rate,
            target_sr=LIVE_SAMPLE_RATE
        )

        data = data.astype(
            np.float32
        )


    return data


# ============================================================
# VAD
# ============================================================

def calculate_rms_v5(audio):

    if audio is None or len(audio) == 0:

        return 0.0


    return float(
        np.sqrt(
            np.mean(
                np.square(
                    audio
                )
            )
        )
    )


def is_speech_v5(audio):

    if audio is None or len(audio) == 0:

        return False


    return (
        calculate_rms_v5(audio)
        > LIVE_RMS_THRESHOLD
    )


# ============================================================
# INDICCONFORMER
# ============================================================

def run_asr_v5(audio):

    if audio is None:
        return ""


    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(audio) < minimum_samples:

        return ""


    wav = torch.from_numpy(
        audio
    ).float().unsqueeze(0).to(DEVICE)


    with torch.inference_mode():

        text = model(
            wav,
            "hi",
            "rnnt"
        )


    if text is None:

        return ""


    return str(
        text
    ).strip()


# ============================================================
# TEXT NORMALIZATION FOR MATCHING
# ============================================================

def normalize_text_for_matching(text):

    if not text:

        return []

    return (
        text
        .strip()
        .split()
    )


# ============================================================
# FIND TEXTUAL OVERLAP
# ============================================================
#
# Example:
#
# Previous:
#
#   "आज मौसम बहुत अच्छा है"
#
# Current:
#
#   "बहुत अच्छा है और मैं बाहर"
#
# Shared suffix/prefix:
#
#   "बहुत अच्छा है"
#
# Result:
#
#   "आज मौसम बहुत अच्छा है और मैं बाहर"
#
# ============================================================

def find_overlap_words(
    previous_text,
    current_text
):

    previous_words = (
        normalize_text_for_matching(
            previous_text
        )
    )

    current_words = (
        normalize_text_for_matching(
            current_text
        )
    )


    if not previous_words or not current_words:

        return 0


    max_overlap = min(
        len(previous_words),
        len(current_words)
    )


    # Prefer the largest suffix/prefix match.

    for overlap in range(
        max_overlap,
        0,
        -1
    ):

        if (
            previous_words[-overlap:]
            ==
            current_words[:overlap]
        ):

            return overlap


    return 0


# ============================================================
# MERGE TWO TRANSCRIPTS
# ============================================================

def merge_transcripts(
    previous_text,
    current_text
):

    previous_text = (
        previous_text
        .strip()
    )

    current_text = (
        current_text
        .strip()
    )


    if not previous_text:

        return current_text


    if not current_text:

        return previous_text


    overlap = find_overlap_words(
        previous_text,
        current_text
    )


    current_words = (
        normalize_text_for_matching(
            current_text
        )
    )


    new_words = current_words[
        overlap:
    ]


    if not new_words:

        return previous_text


    return (
        previous_text
        + " "
        + " ".join(new_words)
    )


# ============================================================
# DISPLAY
# ============================================================

def display_v5(state):

    committed = (
        state["committed_transcript"]
        .strip()
    )

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if committed and partial:

        return (
            committed
            + " "
            + partial
        )


    if committed:

        return committed


    return partial


# ============================================================
# COMMIT CURRENT UTTERANCE
# ============================================================

def commit_current_utterance_v5(
    state
):

    partial = (
        state["partial_transcript"]
        .strip()
    )


    if partial:

        if state["committed_transcript"]:

            state["committed_transcript"] += (
                " "
                + partial
            )

        else:

            state["committed_transcript"] = (
                partial
            )


    # --------------------------------------------------------
    # Reset utterance-specific state
    # --------------------------------------------------------

    state["audio"] = np.zeros(
        0,
        dtype=np.float32
    )

    state["partial_transcript"] = ""

    state["last_window_transcript"] = ""

    state["silence_duration"] = 0.0

    state["new_audio_since_asr"] = 0.0

    state["window_count"] = 0

    state["last_window_start"] = 0.0

    state["utterance_count"] += 1


# ============================================================
# GET CURRENT INFERENCE WINDOW
# ============================================================

def get_current_window_v5(
    audio
):

    if audio is None or len(audio) == 0:

        return audio


    max_samples = int(
        WINDOW_SECONDS
        * LIVE_SAMPLE_RATE
    )


    if len(audio) <= max_samples:

        return audio


    return audio[
        -max_samples:
    ]


# ============================================================
# LIVE CALLBACK
# ============================================================

def live_transcription_v5(
    audio,
    state
):

    # --------------------------------------------------------
    # Initialize
    # --------------------------------------------------------

    if state is None:

        state = make_live_state_v5()


    # --------------------------------------------------------
    # No audio
    # --------------------------------------------------------

    if audio is None:

        return (
            display_v5(state),
            state
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    chunk = normalize_audio_v5(
        audio
    )


    if chunk is None or len(chunk) == 0:

        return (
            display_v5(state),
            state
        )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    state["chunk_count"] += 1


    chunk_duration = (
        len(chunk)
        / LIVE_SAMPLE_RATE
    )


    # --------------------------------------------------------
    # VAD
    # --------------------------------------------------------

    speech = is_speech_v5(
        chunk
    )


    # ========================================================
    # SPEECH
    # ========================================================

    if speech:

        state["silence_duration"] = 0.0


        state["audio"] = np.concatenate(
            [
                state["audio"],
                chunk
            ]
        )


        state["new_audio_since_asr"] += (
            chunk_duration
        )


    # ========================================================
    # SILENCE
    # ========================================================

    else:

        if len(state["audio"]) > 0:

            state["silence_duration"] += (
                chunk_duration
            )


    # ========================================================
    # ASR WINDOW
    # ========================================================

    minimum_samples = int(
        LIVE_MIN_AUDIO_SECONDS
        * LIVE_SAMPLE_RATE
    )


    enough_audio = (
        len(state["audio"])
        >= minimum_samples
    )


    enough_new_audio = (
        state["new_audio_since_asr"]
        >= LIVE_NEW_AUDIO_FOR_ASR
    )


    should_run_asr = (
        speech
        and
        enough_audio
        and
        enough_new_audio
    )


    if should_run_asr:

        # ----------------------------------------------------
        # Extract latest window.
        #
        # The window overlaps previous inference windows.
        # ----------------------------------------------------

        current_window = (
            get_current_window_v5(
                state["audio"]
            )
        )


        try:

            current_text = run_asr_v5(
                current_window
            )


            if current_text:

                # ------------------------------------------------
                # IMPORTANT:
                #
                # We compare the NEW model output with the
                # PREVIOUS model output and remove their shared
                # textual prefix/suffix.
                #
                # This prevents naive duplication.
                # ------------------------------------------------

                previous_window_text = (
                    state[
                        "last_window_transcript"
                    ]
                )


                merged_text = merge_transcripts(
                    previous_window_text,
                    current_text
                )


                # ------------------------------------------------
                # The merged result becomes the current partial.
                # ------------------------------------------------

                state["partial_transcript"] = (
                    merged_text
                )


                state[
                    "last_window_transcript"
                ] = current_text


            state["asr_count"] += 1

            state["window_count"] += 1

            state["new_audio_since_asr"] = 0.0


        except Exception as e:

            return (
                f"ASR ERROR: "
                f"{type(e).__name__}: {e}",
                state
            )


    # ========================================================
    # COMMIT AFTER SILENCE
    # ========================================================

    if (
        state["silence_duration"]
        >= LIVE_COMMIT_SILENCE
    ):

        commit_current_utterance_v5(
            state
        )


    # ========================================================
    # RETURN
    # ========================================================

    return (
        display_v5(state),
        state
    )


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="IndicConformer Live Transcription — Step 5"
) as live_demo_v5:

    gr.Markdown(
        """
        # 🎙️ Live IndicConformer Transcription

        ### Step 5 — Overlapping Windows

        Long continuous speech is handled using overlapping
        inference windows.

        Previous committed speech remains untouched.
        """
    )


    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="🎤 Microphone"
    )


    transcript = gr.Textbox(
        label="Live Transcript",
        lines=12,
        interactive=False
    )


    live_state = gr.State(
        make_live_state_v5()
    )


    microphone.stream(
        fn=live_transcription_v5,
        inputs=[
            microphone,
            live_state
        ],
        outputs=[
            transcript,
            live_state
        ],
        stream_every=LIVE_STREAM_EVERY,
    )


# ============================================================
# LAUNCH
# ============================================================

live_demo_v5.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0459de16c30ae092f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0459de16c30ae092f3.gradio.live
